# YOLOv8 Defect Detection — Kaggle Training
**Zaalima Development · Project 2 · Computer Vision**

Run All → wait ~30 mins → download `best.pt`

In [ ]:
# Install
!pip install -q ultralytics albumentations
import os
from pathlib import Path
print('✅ Installed')

In [ ]:
# Download NEU dataset from Kaggle
# Make sure you have your kaggle.json API key uploaded
!kaggle datasets download -d kaustubhdikshit/neu-surface-defect-database
!unzip -q neu-surface-defect-database.zip -d /kaggle/working/NEU-DET
print('✅ Dataset downloaded')

In [ ]:
import os, shutil, random, cv2
import numpy as np
from pathlib import Path
import albumentations as A

CLASSES = ['crazing','inclusion','patches','pitted_surface','rolled-in_scale','scratches']
BASE    = Path('/kaggle/working')
RAW     = BASE / 'NEU-DET'

for split in ['train','val','test']:
    (BASE/split/'images').mkdir(parents=True, exist_ok=True)
    (BASE/split/'labels').mkdir(parents=True, exist_ok=True)

random.seed(42)
for cls_id, cls in enumerate(CLASSES):
    imgs = sorted((RAW/'IMAGES').glob(f'{cls}_*.jpg'))
    if not imgs:
        imgs = sorted((RAW/'IMAGES').glob(f'{cls}*.jpg'))
    random.shuffle(imgs)
    n = len(imgs)
    splits = [('train', imgs[:int(n*0.8)]),
               ('val',   imgs[int(n*0.8):int(n*0.95)]),
               ('test',  imgs[int(n*0.95):])]
    for split, split_imgs in splits:
        for img in split_imgs:
            shutil.copy2(img, BASE/split/'images'/img.name)
            lbl = BASE/split/'labels'/img.with_suffix('.txt').name
            lbl.write_text(f'{cls_id} 0.5 0.5 0.9 0.9')
    print(f'{cls}: {n} images split')

print('✅ Dataset organised')

In [ ]:
# Augmentation
transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(p=0.6),
    A.GaussNoise(p=0.4),
    A.CLAHE(p=0.4),
    A.Rotate(limit=15, p=0.5),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels'], min_visibility=0.3))

aug_img_dir = BASE/'aug'/'images'
aug_lbl_dir = BASE/'aug'/'labels'
aug_img_dir.mkdir(parents=True, exist_ok=True)
aug_lbl_dir.mkdir(parents=True, exist_ok=True)

for img_path in sorted((BASE/'train'/'images').glob('*.jpg')):
    img = cv2.imread(str(img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    lbl_path = BASE/'train'/'labels'/img_path.with_suffix('.txt').name
    lines = lbl_path.read_text().strip().splitlines()
    cls_ids, bboxes = [], []
    for line in lines:
        p = line.split()
        cls_ids.append(int(p[0]))
        bboxes.append([float(x) for x in p[1:]])

    # Copy original
    shutil.copy2(img_path, aug_img_dir/img_path.name)
    shutil.copy2(lbl_path, aug_lbl_dir/lbl_path.name)

    for i in range(4):
        try:
            r = transform(image=img, bboxes=bboxes, class_labels=cls_ids)
            out = cv2.cvtColor(r['image'], cv2.COLOR_RGB2BGR)
            name = f"{img_path.stem}_aug{i}"
            cv2.imwrite(str(aug_img_dir/f'{name}.jpg'), out)
            lbl_lines = [f"{c} {' '.join(f'{v:.6f}' for v in b)}" for c,b in zip(r['class_labels'], r['bboxes'])]
            (aug_lbl_dir/f'{name}.txt').write_text('\n'.join(lbl_lines))
        except: pass

print(f'✅ Augmentation done. Total images: {len(list(aug_img_dir.glob("*.jpg")))}')

In [ ]:
# Write dataset YAML
yaml = f"""path: /kaggle/working
train: aug/images
val: val/images
test: test/images
nc: 6
names: ['crazing','inclusion','patches','pitted_surface','rolled-in_scale','scratches']
"""
Path('/kaggle/working/neu.yaml').write_text(yaml)
print('✅ YAML written')

In [ ]:
# TRAIN!
from ultralytics import YOLO
model = YOLO('yolov8n.pt')
results = model.train(
    data='/kaggle/working/neu.yaml',
    epochs=80,
    imgsz=200,
    batch=32,
    name='neu_defect',
    project='/kaggle/working/runs',
    patience=20,
    optimizer='AdamW',
    lr0=0.001,
    augment=True,
    plots=True,
    save=True,
)
print(f"✅ Training done! mAP@0.5: {results.results_dict.get('metrics/mAP50(B)', 0):.4f}")

In [ ]:
# Validate
best = YOLO('/kaggle/working/runs/neu_defect/weights/best.pt')
val  = best.val(data='/kaggle/working/neu.yaml')
print(f'mAP@0.5     : {val.box.map50:.4f}')
print(f'mAP@0.5:0.95: {val.box.map:.4f}')
print(f'Precision   : {val.box.mp:.4f}')
print(f'Recall      : {val.box.mr:.4f}')

In [ ]:
# Export to ONNX
best.export(format='onnx', imgsz=200, simplify=True)
print('✅ ONNX exported')
print('📥 Download best.pt from: /kaggle/working/runs/neu_defect/weights/best.pt')